In [1]:
import torch

# Define a list of device types to check
device_types = ['cuda', 'cpu', 'xpu', 'mtia']

# Loop through and check if autocast is available for each
for device_type in device_types:
    is_available = torch.amp.autocast_mode.is_autocast_available(device_type)
    print(f"Autocast available on {device_type}: {is_available}")


Autocast available on cuda: True
Autocast available on cpu: True
Autocast available on xpu: True
Autocast available on mtia: False


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Dummy model definition
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.linear = nn.Linear(10, 1)

    def forward(self, x):
        return self.linear(x)

# Create dummy data
x = torch.randn(64, 10)
y = torch.randn(64, 1)
dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size=16)

# Move model to CUDA
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Net().to(device)

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop with autocast
for epoch in range(2):  # just 2 epochs for demo
    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        # Enable autocast for the forward pass
        with torch.autocast(device_type=device):
            output = model(batch_x)
            loss = loss_fn(output, batch_y)

        # Exit autocast before backward
        loss.backward()
        optimizer.step()

        print(f"Loss: {loss.item():.4f}")


Loss: 1.8584
Loss: 0.4153
Loss: 1.9787
Loss: 1.1122
Loss: 1.7745
Loss: 0.3835
Loss: 1.8815
Loss: 1.0940
